# TP1 — Enrichissement géographique du corpus CAMille avec l’API Nominatim

Ce notebook interroge l’API Nominatim d’OpenStreetMap afin d’obtenir les coordonnées géographiques de lieux susceptibles d’apparaître dans le corpus historique CAMille.

L’objectif est de montrer comment une API peut enrichir des lieux extraits d’un corpus avec des données géographiques structurées.

## Imports

In [1]:
import time

import pandas as pd
import requests

## Interrogation de l’API Nominatim

In [2]:
def geocode_place(place):
    """Retourne le premier résultat géographique fourni par Nominatim."""
    
    url = "https://nominatim.openstreetmap.org/search"
    headers = {
        "User-Agent": "TAC-CAMille-ULB/1.0 (student project)"
    }
    params = {
        "q": place,
        "format": "jsonv2",
        "limit": 1,
        "countrycodes": "be"
    }

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=30
    )
    response.raise_for_status()
    results = response.json()

    if not results:
        return None

    result = results[0]
    return {
        "lieu_recherche": place,
        "nom_complet": result["display_name"],
        "latitude": float(result["lat"]),
        "longitude": float(result["lon"]),
        "type": result.get("type")
    }

## Enrichissement d’une sélection de lieux liés à CAMille

In [3]:
places = [
    "Bruxelles",
    "Anvers",
    "Liège",
    "Bastogne",
    "Vielsalm"
]
geocoded_places = []

for place in places:
    result = geocode_place(place)

    if result is not None:
        geocoded_places.append(result)
        print(f"Trouvé : {place}")
    else:
        print(f"Non trouvé : {place}")

    time.sleep(1)

Trouvé : Bruxelles
Trouvé : Anvers
Trouvé : Liège
Trouvé : Bastogne
Trouvé : Vielsalm


In [4]:
locations_df = pd.DataFrame(geocoded_places)
locations_df

,lieu_recherche,nom_complet,latitude,longitude,type
0,Bruxelles,"Bruxelles - Brussel, Brussel-Hoofdstad - Bruxe...",50.846737,4.352493,administrative
1,Anvers,"Antwerpen, Vlaanderen, België / Belgique / Bel...",51.221110,4.399708,administrative
2,Liège,"Liège, Wallonie, 4000, België / Belgique / Bel...",50.645094,5.573611,administrative
3,Bastogne,"Bastogne, Luxembourg, Wallonie, België / Belgi...",50.002310,5.717339,administrative
4,Vielsalm,"Vielsalm, Bastogne, Luxembourg, Wallonie, Belg...",50.284215,5.916278,administrative


## Interprétation

L’API Nominatim permet d’associer automatiquement des lieux mentionnés dans un corpus à des coordonnées géographiques et à une dénomination normalisée. Cet enrichissement pourrait servir à cartographier les lieux cités dans CAMille ou à comparer leur présence selon les périodes.

Les résultats doivent cependant être contrôlés, car un même nom peut désigner plusieurs lieux. La restriction à la Belgique réduit cette ambiguïté, mais elle exclurait les lieux étrangers présents dans le corpus.

In [5]:
assert len(locations_df) == len(places)
assert locations_df[["latitude", "longitude"]].notna().all().all()

print("Enrichissement géographique terminé et vérifié avec succès.")

Enrichissement géographique terminé et vérifié avec succès.
